# K-means: novidade e descoberta temática

Execução reproduzível do plano em `machine-learning/docs/kmeans.md` e do
protocolo em `machine-learning/docs/comparison-protocol.md`. A trilha de
novidade treina apenas em True do treino; a trilha temática aprende clusters
sem labels. O texto fica somente em memória e não é escrito nos artefatos.

In [1]:
from __future__ import annotations

import hashlib
import json
import os
import platform
import re
import tempfile
import unicodedata
import urllib.parse
import urllib.request
from collections import Counter, defaultdict
from datetime import datetime, timezone
from importlib.metadata import version
from pathlib import Path, PurePosixPath
from zipfile import ZipFile

import numpy as np
import pandas as pd
import scipy
from scipy.sparse import hstack
from sklearn.cluster import KMeans
from sklearn.ensemble import IsolationForest
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    adjusted_rand_score,
    average_precision_score,
    balanced_accuracy_score,
    f1_score,
    normalized_mutual_info_score,
    precision_score,
    recall_score,
    roc_auc_score,
    silhouette_score,
)
from sklearn.neighbors import LocalOutlierFactor
from sklearn.preprocessing import StandardScaler
from sklearn.svm import OneClassSVM

RANDOM_STATE = 42
CORPUS_REVISION = "780f5516c4ae070761632d98ac3368f3ded09d35"
CORPUS_URL = f"https://codeload.github.com/roneysco/Fake.br-Corpus/zip/{CORPUS_REVISION}"
HISTORICAL_ARCHIVE_SHA256 = "be91c188f621424017bd79a0f33528dcc27f8a6151adb2bcb899c17719eb4090"
EXPECTED_METADATA_LINES = 25
CHARACTER_LIMIT = 300
TEXT_WORD_LIMIT = 200
N_SPLITS = 5
TOPIC_K_CANDIDATES = (2, 4, 8, 16)
NOVELTY_K_CANDIDATES = (1, 2, 4, 8)
LOF_NEIGHBORS = (10, 20, 40, 80)
OCSVM_NU = (0.01, 0.025, 0.05, 0.10)
KMEANS_N_INIT = 20
TOPIC_N_INIT = 10
SELECTION_TOLERANCE = 1e-12
SILHOUETTE_SAMPLE_SIZE = 1000
STABILITY_SEEDS = (42, 17, 123)
STYLE_FEATURES = [
    "tem_autor",
    "typeTokenRatio",
    "linkDensity",
    "punctuationDensity",
    "uppercaseRatio",
    "diversidade",
]
WORD_PATTERN = re.compile(r"[^\W\d_]+(?:['’\-][^\W\d_]+)*", re.UNICODE)
TOKEN_PATTERN = re.compile(
    r"[^\W\d_]+(?:['’\-][^\W\d_]+)*|\d+(?:[.,]\d+)*|[^\w\s]",
    re.UNICODE,
)

## Fonte e corpus

A revisão é fixa. O ZIP é mantido em cache temporário fora do repositório;
seu SHA-256 calculado nesta execução é registrado no manifesto.

In [2]:
project_root = Path.cwd().resolve()
while project_root != project_root.parent and not (project_root / ".git").exists():
    project_root = project_root.parent
if not (project_root / ".git").exists():
    raise RuntimeError("Execute a partir do repositório olimpo-fake-news-ai.")

archive_cache = Path(tempfile.gettempdir()) / "olimpo-fake-news-ai" / "corpus"
archive_cache.mkdir(parents=True, exist_ok=True)
archive_path = archive_cache / f"Fake.br-Corpus-{CORPUS_REVISION}.zip"
if not archive_path.exists():
    temporary_archive = archive_path.with_suffix(".download")
    request = urllib.request.Request(CORPUS_URL, headers={"User-Agent": "kmeans-experiment/1.0"})
    with urllib.request.urlopen(request, timeout=180) as response, temporary_archive.open("wb") as target:
        while chunk := response.read(1024 * 1024):
            target.write(chunk)
    temporary_archive.replace(archive_path)

archive_sha256 = hashlib.sha256(archive_path.read_bytes()).hexdigest()
with ZipFile(archive_path) as archive:
    bad_member = archive.testzip()
    if bad_member is not None:
        raise RuntimeError(f"ZIP corrompido no membro {bad_member!r}.")
    archive_names = archive.namelist()

print("Revisão do corpus:", CORPUS_REVISION)
print("URL:", CORPUS_URL)
print("SHA-256 calculado:", archive_sha256)
print("Confere com o hash registrado anteriormente:", archive_sha256 == HISTORICAL_ARCHIVE_SHA256)

Revisão do corpus: 780f5516c4ae070761632d98ac3368f3ded09d35
URL: https://codeload.github.com/roneysco/Fake.br-Corpus/zip/780f5516c4ae070761632d98ac3368f3ded09d35
SHA-256 calculado: be91c188f621424017bd79a0f33528dcc27f8a6151adb2bcb899c17719eb4090
Confere com o hash registrado anteriormente: True


## Carregamento, IDs e grupos

`record_id` usa classe de origem e nome do arquivo. `group_id` une o par
Fake/True alinhado pelo mesmo nome e também textos integralmente duplicados
após normalização Unicode e de espaços. O grupo nunca entra como feature.

In [3]:
metadata_columns = [
    "autor", "link", "categoria", "data_publicacao", "num_tokens", "num_palavras",
    "num_types", "num_links", "num_maiusculas", "num_verbos", "num_verbos_subj_imp",
    "num_substantivos", "num_adjetivos", "num_adverbios", "num_verbos_modais",
    "num_pron_1_2_sing", "num_pron_1_plural", "num_pronomes", "pausalidade",
    "num_caracteres", "tam_medio_sentenca", "tam_medio_palavra", "pct_erros_ortograficos",
    "emotividade", "diversidade",
]


def members_for(archive, directory: str, metadata: bool = False):
    suffix = "-meta.txt" if metadata else ".txt"
    marker = f"/full_texts/{directory}/"
    found = {}
    for name in archive.namelist():
        if marker not in name or not name.endswith(suffix):
            continue
        stem = PurePosixPath(name).name.removesuffix(suffix)
        if stem in found:
            raise ValueError(f"Nome de arquivo repetido em {directory}: {stem}")
        found[stem] = name
    return found


records = []
with ZipFile(archive_path) as archive:
    archive_index = set(archive.namelist())
    for source_folder, label in (("fake", 1), ("true", 0)):
        text_members = members_for(archive, source_folder)
        meta_members = members_for(archive, f"{source_folder}-meta-information", metadata=True)
        if set(text_members) != set(meta_members):
            raise ValueError(
                f"Texto/metadados não correspondem em {source_folder}: "
                f"{len(set(text_members) - set(meta_members))} sem metadado; "
                f"{len(set(meta_members) - set(text_members))} sem texto."
            )
        for stem in sorted(text_members):
            text_path = text_members[stem]
            metadata_path = meta_members[stem]
            if text_path not in archive_index or metadata_path not in archive_index:
                raise ValueError("Caminho do corpus ausente no ZIP.")
            text = archive.read(text_path).decode("utf-8")
            metadata = archive.read(metadata_path).decode("utf-8").splitlines()
            if len(metadata) != EXPECTED_METADATA_LINES:
                raise ValueError(
                    f"Schema inesperado em {metadata_path}: {len(metadata)} linhas; "
                    f"esperado {EXPECTED_METADATA_LINES}."
                )
            records.append({
                "record_id": f"{source_folder}/{stem}",
                "pair_key": stem,
                "label": label,
                "text": text,
                "author": metadata[0],
                "link": metadata[1],
                "publication_date": metadata[3],
            })

news = pd.DataFrame.from_records(records)
if news["record_id"].duplicated().any() or news.empty:
    raise ValueError("record_id deve ser único e o corpus não pode estar vazio.")
if set(news["label"].unique()) != {0, 1}:
    raise ValueError("Labels fora do protocolo: 0=True e 1=Fake.")
if not news.loc[news["record_id"].str.startswith("true/"), "label"].eq(0).all():
    raise ValueError("A classe True precisa permanecer como label 0.")
if not news.loc[news["record_id"].str.startswith("fake/"), "label"].eq(1).all():
    raise ValueError("A classe Fake precisa permanecer como label 1.")


class UnionFind:
    def __init__(self, items):
        self.parent = {item: item for item in items}

    def find(self, item):
        while self.parent[item] != item:
            self.parent[item] = self.parent[self.parent[item]]
            item = self.parent[item]
        return item

    def union(self, left, right):
        left_root, right_root = self.find(left), self.find(right)
        if left_root != right_root:
            first, second = sorted((left_root, right_root))
            self.parent[second] = first


union_find = UnionFind(news["record_id"].tolist())
by_pair = defaultdict(list)
by_text_hash = defaultdict(list)
for row in news.itertuples(index=False):
    by_pair[row.pair_key].append(row.record_id)
    normalized_full_text = " ".join(
        unicodedata.normalize("NFKC", row.text).lstrip("\ufeff").split()
    ).casefold()
    by_text_hash[hashlib.sha256(normalized_full_text.encode("utf-8")).hexdigest()].append(row.record_id)
for item_group in list(by_pair.values()) + list(by_text_hash.values()):
    if len(item_group) > 1:
        for record_id in item_group[1:]:
            union_find.union(item_group[0], record_id)
components = defaultdict(list)
for record_id in news["record_id"]:
    components[union_find.find(record_id)].append(record_id)
group_id_for_record = {}
for component in components.values():
    component_key = "|".join(sorted(component)).encode("utf-8")
    group_id = "group-" + hashlib.sha256(component_key).hexdigest()[:16]
    group_id_for_record.update({record_id: group_id for record_id in component})
news["group_id"] = news["record_id"].map(group_id_for_record)
news["texto_trunc"] = news["text"].map(lambda value: " ".join(value.split()[:TEXT_WORD_LIMIT]))

fake_stems = set(news.loc[news["label"].eq(1), "pair_key"])
true_stems = set(news.loc[news["label"].eq(0), "pair_key"])
aligned_pair_stems = fake_stems & true_stems
print(f"Corpus carregado: {len(news):,} notícias; True={int((news.label == 0).sum()):,}; Fake={int((news.label == 1).sum()):,}.")
print(f"Grupos de pares Fake/True alinhados: {len(aligned_pair_stems):,}; grupos totais após deduplicação exata: {news.group_id.nunique():,}.")

Corpus carregado: 7,200 notícias; True=3,600; Fake=3,600.
Grupos de pares Fake/True alinhados: 3,600; grupos totais após deduplicação exata: 3,599.


## Features de estilo e partição canônica

O primeiro fold é teste, o segundo validação e os três restantes treino de
`StratifiedGroupKFold` (60/20/20 aproximado). Os labels só estratificam os
folds; os `group_id` mantêm pares/deduplicatas juntos. A autoria fica binária
e sem padronização, limitando sua contribuição à distância a 0 ou 1.

In [4]:
def finite_ratio(numerator, denominator):
    if denominator <= 0 or not np.isfinite(denominator):
        return np.nan
    return float(numerator / denominator)


def style_from_row(row):
    normalized = unicodedata.normalize("NFKC", row.text).lstrip("\ufeff")[:CHARACTER_LIMIT]
    words = WORD_PATTERN.findall(normalized)
    tokens = TOKEN_PATTERN.findall(normalized)
    n_words, n_tokens = len(words), len(tokens)
    n_types = len({word.casefold() for word in words})
    n_uppercase = sum(word.isupper() and len(word) > 1 for word in words)
    n_links = len(re.findall(r"https?://\S+", normalized, flags=re.IGNORECASE))
    author_value = str(row.author).strip().casefold()
    has_author = int(author_value not in {"", "none", "null", "nan"})
    return {
        "tem_autor": has_author,
        "typeTokenRatio": finite_ratio(n_types, n_tokens),
        "linkDensity": finite_ratio(n_links, n_words),
        "punctuationDensity": finite_ratio(n_tokens - n_words, n_tokens),
        "uppercaseRatio": finite_ratio(n_uppercase, n_words),
        "diversidade": finite_ratio(n_types, n_words),
    }


style_features = pd.DataFrame([style_from_row(row) for row in news.itertuples(index=False)])
style_features = style_features[STYLE_FEATURES].replace([np.inf, -np.inf], np.nan)
news = pd.concat([news.reset_index(drop=True), style_features], axis=1)

from sklearn.model_selection import StratifiedGroupKFold

splitter = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
folds = list(splitter.split(news["record_id"], news["label"], groups=news["group_id"]))
test_indices = folds[0][1]
validation_indices = folds[1][1]
test_groups = set(news.iloc[test_indices]["group_id"])
validation_groups = set(news.iloc[validation_indices]["group_id"])
train_indices = np.asarray([
    index for index, group_id in enumerate(news["group_id"])
    if group_id not in test_groups and group_id not in validation_groups
], dtype=int)
partition_indices = {
    "train": train_indices,
    "validation": validation_indices,
    "test": test_indices,
}
partition_ids = {
    name: set(news.iloc[indices]["record_id"])
    for name, indices in partition_indices.items()
}
partition_groups = {
    name: set(news.iloc[indices]["group_id"])
    for name, indices in partition_indices.items()
}
for left, right in (("train", "validation"), ("train", "test"), ("validation", "test")):
    if partition_ids[left] & partition_ids[right] or partition_groups[left] & partition_groups[right]:
        raise AssertionError(f"Partições sobrepostas: {left}/{right}.")
if set.union(*partition_ids.values()) != set(news["record_id"]):
    raise AssertionError("As partições não cobrem exatamente todos os record_id.")

train_frame = news.iloc[train_indices].reset_index(drop=True)
validation_frame = news.iloc[validation_indices].reset_index(drop=True)
true_train_frame = train_frame.loc[train_frame["label"].eq(0)].reset_index(drop=True)
true_validation_mask = validation_frame["label"].eq(0).to_numpy()
if true_train_frame.empty or not true_train_frame["label"].eq(0).all():
    raise AssertionError("Ajuste de novidade permitido somente com True do treino.")

partition_summary = pd.DataFrame([
    {
        "partition": name,
        "records": len(indices),
        "groups": len(partition_groups[name]),
    }
    for name, indices in partition_indices.items()
]).set_index("partition")
print("Partição canônica por grupo (seed=42):")
print(partition_summary.to_string())

# Auditoria de disponibilidade para um protocolo secundário temporal/fonte.
link_hosts = news["link"].fillna("").map(lambda value: urllib.parse.urlparse(str(value)).hostname or "")
news["source_domain_audit"] = link_hosts.str.lower().str.removeprefix("www.")
PORTUGUESE_MONTHS = {
    "janeiro": 1, "fevereiro": 2, "marco": 3, "abril": 4, "maio": 5, "junho": 6,
    "julho": 7, "agosto": 8, "setembro": 9, "outubro": 10, "novembro": 11, "dezembro": 12,
}


def parse_publication_date(value):
    normalized = unicodedata.normalize("NFKD", str(value).strip().casefold())
    normalized = "".join(character for character in normalized if not unicodedata.combining(character))
    month_match = re.fullmatch(r"(\d{1,2}) de ([a-z]+) de (\d{4})", normalized)
    if month_match and month_match.group(2) in PORTUGUESE_MONTHS:
        try:
            return pd.Timestamp(datetime(
                int(month_match.group(3)), PORTUGUESE_MONTHS[month_match.group(2)], int(month_match.group(1))
            ), tz="UTC")
        except ValueError:
            return pd.NaT
    for date_format in ("%d/%m/%Y", "%Y-%m-%d"):
        try:
            return pd.Timestamp(datetime.strptime(normalized[:10], date_format), tz="UTC")
        except ValueError:
            continue
    return pd.NaT


news["publication_datetime"] = news["publication_date"].map(parse_publication_date)
date_counts_by_label = {
    str(int(label)): int(group["publication_datetime"].notna().sum())
    for label, group in news.groupby("label", sort=True)
}
domain_counts_by_label = {
    "true_0": int(news.loc[news["label"].eq(0), "source_domain_audit"].replace("", np.nan).nunique()),
    "fake_1": int(news.loc[news["label"].eq(1), "source_domain_audit"].replace("", np.nan).nunique()),
}
true_domains = set(news.loc[news["label"].eq(0), "source_domain_audit"]) - {""}
fake_domains = set(news.loc[news["label"].eq(1), "source_domain_audit"]) - {""}
source_adjacency = defaultdict(set)
for _, source_group in news.groupby("group_id", sort=False):
    domains = sorted(set(source_group["source_domain_audit"]) - {""})
    for domain in domains:
        source_adjacency[domain].update(set(domains) - {domain})
source_components = 0
unseen_domains = set(source_adjacency)
while unseen_domains:
    source_components += 1
    pending = [unseen_domains.pop()]
    while pending:
        for neighbor in source_adjacency[pending.pop()] & unseen_domains:
            unseen_domains.remove(neighbor)
            pending.append(neighbor)

group_date_rows = []
for group_id, group in news.groupby("group_id", sort=False):
    parsed = group["publication_datetime"]
    all_dates_present = bool(parsed.notna().all())
    group_date_rows.append({
        "group_id": group_id,
        "group_time": parsed.max() if all_dates_present else pd.NaT,
        "all_dates_present": all_dates_present,
    })
group_date_frame = pd.DataFrame(group_date_rows)
temporal_group_frame = group_date_frame.loc[group_date_frame["all_dates_present"]].sort_values(
    ["group_time", "group_id"], kind="mergesort"
).reset_index(drop=True)
if len(temporal_group_frame) < 1000:
    raise ValueError(
        "Holdout temporal secundário requer pelo menos 1.000 grupos com todas as datas parseáveis; "
        f"foram encontrados {len(temporal_group_frame)}."
    )
temporal_train_boundary = temporal_group_frame.iloc[int(len(temporal_group_frame) * 0.60)]["group_time"]
temporal_test_boundary = temporal_group_frame.iloc[int(len(temporal_group_frame) * 0.80)]["group_time"]
temporal_group_sets = {
    "train": set(temporal_group_frame.loc[
        temporal_group_frame["group_time"].lt(temporal_train_boundary), "group_id"
    ]),
    "validation": set(temporal_group_frame.loc[
        temporal_group_frame["group_time"].ge(temporal_train_boundary)
        & temporal_group_frame["group_time"].lt(temporal_test_boundary), "group_id"
    ]),
    "test": set(temporal_group_frame.loc[
        temporal_group_frame["group_time"].ge(temporal_test_boundary), "group_id"
    ]),
}
temporal_partition_indices = {
    name: np.flatnonzero(news["group_id"].isin(groups).to_numpy())
    for name, groups in temporal_group_sets.items()
}
temporal_partition_ids = {
    name: set(news.iloc[indices]["record_id"])
    for name, indices in temporal_partition_indices.items()
}
for left, right in (("train", "validation"), ("train", "test"), ("validation", "test")):
    if temporal_group_sets[left] & temporal_group_sets[right] or temporal_partition_ids[left] & temporal_partition_ids[right]:
        raise AssertionError(f"Sobreposição na partição temporal: {left}/{right}.")
if not (
    temporal_group_frame.loc[temporal_group_frame["group_id"].isin(temporal_group_sets["train"]), "group_time"].max()
    < temporal_group_frame.loc[temporal_group_frame["group_id"].isin(temporal_group_sets["validation"]), "group_time"].min()
    <= temporal_group_frame.loc[temporal_group_frame["group_id"].isin(temporal_group_sets["validation"]), "group_time"].max()
    < temporal_group_frame.loc[temporal_group_frame["group_id"].isin(temporal_group_sets["test"]), "group_time"].min()
):
    raise AssertionError("As datas do holdout temporal não são estritamente ordenadas por partição.")
temporal_frames = {
    name: news.iloc[temporal_partition_indices[name]].reset_index(drop=True)
    for name in ("train", "validation")
}
temporal_true_train_frame = temporal_frames["train"].loc[
    temporal_frames["train"]["label"].eq(0)
].reset_index(drop=True)
if any(not frame["label"].isin([0, 1]).all() for frame in temporal_frames.values()):
    raise AssertionError("Label fora do protocolo temporal.")

secondary_audit = {
    "records_with_source_domain": int(news["source_domain_audit"].ne("").sum()),
    "distinct_source_domains": int(news.loc[news["source_domain_audit"].ne(""), "source_domain_audit"].nunique()),
    "source_domains_by_label": domain_counts_by_label,
    "common_source_domains_between_labels": len(true_domains & fake_domains),
    "source_graph_components_when_aligned_pairs_are_edges": int(source_components),
    "records_with_parseable_publication_date": int(news["publication_datetime"].notna().sum()),
    "records_with_parseable_date_by_label": date_counts_by_label,
    "groups_with_all_dates_parseable": int(len(temporal_group_frame)),
    "groups_excluded_for_missing_or_invalid_date": int(news["group_id"].nunique() - len(temporal_group_frame)),
    "distinct_publication_years": sorted(
        int(year) for year in temporal_group_frame["group_time"].dt.year.unique()
    ),
    "date_parse_formats": ["DD/MM/YYYY", "YYYY-MM-DD", "D de mês_pt de YYYY"],
}
secondary_temporal_summary = pd.DataFrame([
    {
        "partition": name,
        "records": int(len(temporal_partition_indices[name])),
        "groups": int(len(temporal_group_sets[name])),
        "first_group_date_utc": temporal_group_frame.loc[
            temporal_group_frame["group_id"].isin(temporal_group_sets[name]), "group_time"
        ].min().isoformat(),
        "last_group_date_utc": temporal_group_frame.loc[
            temporal_group_frame["group_id"].isin(temporal_group_sets[name]), "group_time"
        ].max().isoformat(),
    }
    for name in ("train", "validation", "test")
])
print("Auditoria de cobertura para holdout por fonte/período:", secondary_audit)
print("Partição temporal secundária por data do grupo (max das datas Fake/True):")
print(secondary_temporal_summary.to_string(index=False))

Partição canônica por grupo (seed=42):
            records  groups
partition                  
train          4320    2159
validation     1440     720
test           1440     720


Auditoria de cobertura para holdout por fonte/período: {'records_with_source_domain': 7200, 'distinct_source_domains': 29, 'source_domains_by_label': {'true_0': 24, 'fake_1': 5}, 'common_source_domains_between_labels': 0, 'source_graph_components_when_aligned_pairs_are_edges': 1, 'records_with_parseable_publication_date': 7199, 'records_with_parseable_date_by_label': {'0': 3599, '1': 3600}, 'groups_with_all_dates_parseable': 3598, 'groups_excluded_for_missing_or_invalid_date': 1, 'distinct_publication_years': [2015, 2016, 2017, 2018], 'date_parse_formats': ['DD/MM/YYYY', 'YYYY-MM-DD', 'D de mês_pt de YYYY']}
Partição temporal secundária por data do grupo (max das datas Fake/True):
 partition  records  groups      first_group_date_utc       last_group_date_utc
     train     4304    2151 2015-10-13T00:00:00+00:00 2017-12-13T00:00:00+00:00
validation     1452     726 2017-12-14T00:00:00+00:00 2018-02-07T00:00:00+00:00
      test     1442     721 2018-02-08T00:00:00+00:00 2018-07-23T00:00

## Pré-processamento e comparação de novidade

Regra declarada antes do teste: maximizar ROC-AUC na validação; desempatar
por AP e depois pelo menor `k` (ou menor hiperparâmetro numérico). Cada corte
é q95 dos scores de True-validation. O pipeline inteiro ajusta-se somente em
True-train. A mesma matriz canônica de estilo é usada nos quatro detectores.

In [5]:
imputer = SimpleImputer(strategy="median")
imputer.fit(true_train_frame[STYLE_FEATURES])
train_true_imputed = imputer.transform(true_train_frame[STYLE_FEATURES])
continuous_scaler = StandardScaler()
continuous_scaler.fit(train_true_imputed[:, 1:])


def transform_style(frame):
    imputed = imputer.transform(frame[STYLE_FEATURES])
    transformed = np.column_stack((imputed[:, 0], continuous_scaler.transform(imputed[:, 1:])))
    if not np.isfinite(transformed).all():
        raise ValueError("Feature de estilo não finita após imputação/escala.")
    if not np.isin(transformed[:, 0], [0.0, 1.0]).all():
        raise AssertionError("Autoria deixou de ser binária e limitada a [0, 1].")
    return transformed


X_true_train = transform_style(true_train_frame)
X_validation = transform_style(validation_frame)
y_validation = validation_frame["label"].to_numpy(dtype=int)
X_true_validation = X_validation[true_validation_mask]
if X_true_train.shape[0] != len(true_train_frame):
    raise AssertionError("IDs de ajuste True não correspondem às linhas treinadas.")


def anomaly_metrics(labels, scores, threshold):
    labels = np.asarray(labels, dtype=int)
    scores = np.asarray(scores, dtype=float)
    decisions = scores >= float(threshold)
    true_mask = labels == 0
    return {
        "n_samples": int(len(labels)),
        "n_true": int(true_mask.sum()),
        "n_fake": int((labels == 1).sum()),
        "roc_auc": float(roc_auc_score(labels, scores)),
        "average_precision": float(average_precision_score(labels, scores)),
        "macro_f1": float(f1_score(labels, decisions.astype(int), average="macro", zero_division=0)),
        "balanced_accuracy": float(balanced_accuracy_score(labels, decisions.astype(int))),
        "precision_fake": float(precision_score(labels, decisions.astype(int), pos_label=1, zero_division=0)),
        "recall_fake": float(recall_score(labels, decisions.astype(int), pos_label=1, zero_division=0)),
        "f1_fake": float(f1_score(labels, decisions.astype(int), pos_label=1, zero_division=0)),
        "fpr_true": float(decisions[true_mask].mean()) if true_mask.any() else np.nan,
        "accuracy": float(accuracy_score(labels, decisions.astype(int))),
        "threshold_q95": float(threshold),
        "q95_source": "True-validation scores only",
        "validation_true_alert_rate": float(decisions[true_mask].mean()) if true_mask.any() else np.nan,
    }


def select_by_validation(records, parameter_name):
    best_auc = max(record["validation_metrics"]["roc_auc"] for record in records)
    auc_ties = [
        record for record in records
        if best_auc - record["validation_metrics"]["roc_auc"] <= SELECTION_TOLERANCE
    ]
    best_ap = max(record["validation_metrics"]["average_precision"] for record in auc_ties)
    ap_ties = [
        record for record in auc_ties
        if best_ap - record["validation_metrics"]["average_precision"] <= SELECTION_TOLERANCE
    ]
    return min(ap_ties, key=lambda record: record[parameter_name])


novelty_models = {}
novelty_validation_records = []
for candidate_k in NOVELTY_K_CANDIDATES:
    estimator = KMeans(
        n_clusters=candidate_k,
        init="k-means++",
        n_init=KMEANS_N_INIT,
        random_state=RANDOM_STATE,
        algorithm="lloyd",
    )
    estimator.fit(X_true_train)
    validation_distances = estimator.transform(X_validation).min(axis=1)
    threshold = float(np.quantile(validation_distances[true_validation_mask], 0.95))
    metrics = anomaly_metrics(y_validation, validation_distances, threshold)
    candidate = {
        "method": "KMeansNovelty",
        "k": candidate_k,
        "estimator": estimator,
        "scores_validation": validation_distances,
        "threshold": threshold,
        "validation_metrics": metrics,
        "parameters": {
            "n_clusters": candidate_k,
            "init": "k-means++",
            "n_init": KMEANS_N_INIT,
            "random_state": RANDOM_STATE,
            "algorithm": "lloyd",
        },
    }
    novelty_models[("KMeansNovelty", candidate_k)] = candidate
    novelty_validation_records.append(candidate)

isolation_forest = IsolationForest(
    n_estimators=300,
    contamination="auto",
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
isolation_forest.fit(X_true_train)
if_scores_validation = -isolation_forest.decision_function(X_validation)
if_threshold = float(np.quantile(if_scores_validation[true_validation_mask], 0.95))
if_record = {
    "method": "IsolationForest",
    "parameter_value": 300,
    "estimator": isolation_forest,
    "scores_validation": if_scores_validation,
    "threshold": if_threshold,
    "validation_metrics": anomaly_metrics(y_validation, if_scores_validation, if_threshold),
    "parameters": {
        "n_estimators": 300,
        "contamination": "auto",
        "random_state": RANDOM_STATE,
        "n_jobs": -1,
    },
}

lof_records = []
for n_neighbors in LOF_NEIGHBORS:
    estimator = LocalOutlierFactor(
        n_neighbors=n_neighbors,
        algorithm="auto",
        metric="minkowski",
        p=2,
        contamination="auto",
        novelty=True,
        n_jobs=1,
    )
    estimator.fit(X_true_train)
    validation_scores = -estimator.decision_function(X_validation)
    threshold = float(np.quantile(validation_scores[true_validation_mask], 0.95))
    lof_records.append({
        "method": "LOF",
        "n_neighbors": n_neighbors,
        "estimator": estimator,
        "scores_validation": validation_scores,
        "threshold": threshold,
        "validation_metrics": anomaly_metrics(y_validation, validation_scores, threshold),
        "parameters": {
            "n_neighbors": n_neighbors,
            "algorithm": "auto",
            "metric": "minkowski",
            "p": 2,
            "contamination": "auto",
            "novelty": True,
            "n_jobs": 1,
        },
    })

ocsvm_records = []
for nu in OCSVM_NU:
    estimator = OneClassSVM(kernel="rbf", gamma="scale", nu=nu)
    estimator.fit(X_true_train)
    validation_scores = -estimator.decision_function(X_validation)
    threshold = float(np.quantile(validation_scores[true_validation_mask], 0.95))
    ocsvm_records.append({
        "method": "OneClassSVM",
        "nu": nu,
        "estimator": estimator,
        "scores_validation": validation_scores,
        "threshold": threshold,
        "validation_metrics": anomaly_metrics(y_validation, validation_scores, threshold),
        "parameters": {"kernel": "rbf", "gamma": "scale", "nu": nu},
    })

selected_kmeans_novelty = select_by_validation(novelty_validation_records, "k")
selected_lof = select_by_validation(lof_records, "n_neighbors")
selected_ocsvm = select_by_validation(ocsvm_records, "nu")
selected_novelty = {
    "KMeansNovelty": selected_kmeans_novelty,
    "IsolationForest": if_record,
    "LOF": selected_lof,
    "OneClassSVM": selected_ocsvm,
}
for selected in selected_novelty.values():
    selected["selected"] = True

frozen_selection = {
    "selection_rule": "validation ROC-AUC, then validation AP, then lower k/parameter; ties within 1e-12",
    "kmeans_novelty": {
        "k": selected_kmeans_novelty["k"],
        "threshold_q95": selected_kmeans_novelty["threshold"],
        "parameters": selected_kmeans_novelty["parameters"],
        "validation_roc_auc": selected_kmeans_novelty["validation_metrics"]["roc_auc"],
        "validation_average_precision": selected_kmeans_novelty["validation_metrics"]["average_precision"],
    },
    "isolation_forest": {"threshold_q95": if_record["threshold"], "parameters": if_record["parameters"]},
    "lof": {
        "n_neighbors": selected_lof["n_neighbors"],
        "threshold_q95": selected_lof["threshold"],
        "parameters": selected_lof["parameters"],
        "validation_roc_auc": selected_lof["validation_metrics"]["roc_auc"],
        "validation_average_precision": selected_lof["validation_metrics"]["average_precision"],
    },
    "one_class_svm": {
        "nu": selected_ocsvm["nu"],
        "threshold_q95": selected_ocsvm["threshold"],
        "parameters": selected_ocsvm["parameters"],
        "validation_roc_auc": selected_ocsvm["validation_metrics"]["roc_auc"],
        "validation_average_precision": selected_ocsvm["validation_metrics"]["average_precision"],
    },
    "test_opened": False,
}
print("Seleção congelada antes da avaliação de teste:")
print(json.dumps(frozen_selection, ensure_ascii=False, indent=2))

Seleção congelada antes da avaliação de teste:
{
  "selection_rule": "validation ROC-AUC, then validation AP, then lower k/parameter; ties within 1e-12",
  "kmeans_novelty": {
    "k": 8,
    "threshold_q95": 2.030049808893606,
    "parameters": {
      "n_clusters": 8,
      "init": "k-means++",
      "n_init": 20,
      "random_state": 42,
      "algorithm": "lloyd"
    },
    "validation_roc_auc": 0.8351003086419752,
    "validation_average_precision": 0.7503202767489252
  },
  "isolation_forest": {
    "threshold_q95": 0.08418247008884833,
    "parameters": {
      "n_estimators": 300,
      "contamination": "auto",
      "random_state": 42,
      "n_jobs": -1
    }
  },
  "lof": {
    "n_neighbors": 10,
    "threshold_q95": -0.054244918135349506,
    "parameters": {
      "n_neighbors": 10,
      "algorithm": "auto",
      "metric": "minkowski",
      "p": 2,
      "contamination": "auto",
      "novelty": true,
      "n_jobs": 1
    },
    "validation_roc_auc": 0.9766820987654321

## Descoberta temática sem labels

TF-IDF de palavras e caracteres é ajustado somente com `texto_trunc` do
treino. As matrizes permanecem esparsas; somente centroides e matrizes de
distâncias pequenas (n × k) são densas. Escolha de k: maior silhouette
cosseno na validação, desempate pelo menor k. Labels não participam da escolha.

In [6]:
word_vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=2,
    sublinear_tf=True,
    token_pattern=r"(?u)\b[^\d\W]{2,}\b",
    dtype=np.float32,
)
char_vectorizer = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(3, 5),
    min_df=3,
    sublinear_tf=True,
    dtype=np.float32,
)
train_texts = train_frame["texto_trunc"].tolist()
validation_texts = validation_frame["texto_trunc"].tolist()
X_word_train = word_vectorizer.fit_transform(train_texts)
X_char_train = char_vectorizer.fit_transform(train_texts)
X_topic_train = hstack((X_word_train, X_char_train), format="csr", dtype=np.float32)
X_topic_validation = hstack((
    word_vectorizer.transform(validation_texts),
    char_vectorizer.transform(validation_texts),
), format="csr", dtype=np.float32)
if not scipy.sparse.issparse(X_topic_train) or not scipy.sparse.issparse(X_topic_validation):
    raise AssertionError("A matriz TF-IDF deve permanecer esparsa.")
print("TF-IDF shape train/validation:", X_topic_train.shape, X_topic_validation.shape)
print("TF-IDF nnz train/validation:", X_topic_train.nnz, X_topic_validation.nnz)

topic_candidate_records = []
topic_candidate_models = {}
for candidate_k in TOPIC_K_CANDIDATES:
    estimator = KMeans(
        n_clusters=candidate_k,
        init="k-means++",
        n_init=TOPIC_N_INIT,
        random_state=RANDOM_STATE,
        algorithm="lloyd",
    )
    training_cluster_ids = estimator.fit_predict(X_topic_train)
    validation_cluster_ids = estimator.predict(X_topic_validation)
    distinct_validation_clusters = np.unique(validation_cluster_ids).size
    if 1 < distinct_validation_clusters < len(validation_cluster_ids):
        validation_silhouette = float(silhouette_score(
            X_topic_validation,
            validation_cluster_ids,
            metric="cosine",
            sample_size=min(SILHOUETTE_SAMPLE_SIZE, len(validation_cluster_ids)),
            random_state=RANDOM_STATE,
        ))
    else:
        validation_silhouette = float("nan")
    candidate = {
        "k": candidate_k,
        "estimator": estimator,
        "train_cluster_ids": training_cluster_ids,
        "validation_cluster_ids": validation_cluster_ids,
        "validation_silhouette_cosine": validation_silhouette,
        "inertia_train": float(estimator.inertia_),
        "parameters": {
            "n_clusters": candidate_k,
            "init": "k-means++",
            "n_init": TOPIC_N_INIT,
            "random_state": RANDOM_STATE,
            "algorithm": "lloyd",
            "features": "word+char TF-IDF, sparse CSR",
        },
    }
    topic_candidate_models[candidate_k] = candidate
    topic_candidate_records.append(candidate)

valid_topic_candidates = [
    record for record in topic_candidate_records
    if np.isfinite(record["validation_silhouette_cosine"])
]
if not valid_topic_candidates:
    raise ValueError("Nenhum k produziu mais de um cluster previsto na validação.")
selected_topic = max(
    valid_topic_candidates,
    key=lambda record: (record["validation_silhouette_cosine"], -record["k"]),
)
selected_topic["selected"] = True
selected_topic_k = selected_topic["k"]
print("k temático congelado:", selected_topic_k)
print("Silhouette cosseno de validação:", selected_topic["validation_silhouette_cosine"])
print("Inércia no treino:", selected_topic["inertia_train"])

# Estabilidade de atribuições no treino: ajusta só as linhas do treino, sem labels.
topic_train_seed_labels = {RANDOM_STATE: selected_topic["train_cluster_ids"]}
for stability_seed in STABILITY_SEEDS:
    if stability_seed == RANDOM_STATE:
        continue
    stability_estimator = KMeans(
        n_clusters=selected_topic_k,
        init="k-means++",
        n_init=TOPIC_N_INIT,
        random_state=stability_seed,
        algorithm="lloyd",
    )
    topic_train_seed_labels[stability_seed] = stability_estimator.fit_predict(X_topic_train)
stability_pairs = []
for left_seed_index, left_seed in enumerate(STABILITY_SEEDS):
    for right_seed in STABILITY_SEEDS[left_seed_index + 1:]:
        stability_pairs.append({
            "seed_left": left_seed,
            "seed_right": right_seed,
            "ari": float(adjusted_rand_score(
                topic_train_seed_labels[left_seed], topic_train_seed_labels[right_seed]
            )),
        })
stability_mean_ari = float(np.mean([pair["ari"] for pair in stability_pairs]))

TF-IDF shape train/validation: (4320, 163107) (1440, 163107)
TF-IDF nnz train/validation: 6679978 2195818


k temático congelado: 8
Silhouette cosseno de validação: 0.009752699173986912
Inércia no treino: 8037.9970703125


## Avaliação secundária temporal

O protocolo por fonte estrito é inviável: fontes conectadas por pares
alinhados formam um único componente e domínios separados correspondem às
classes. As datas, porém, permitem um segundo teste agrupado. A data do grupo
é a mais recente entre as datas True/Fake; grupos com data inválida são
excluídos. Seleção e q95 são refeitos em treino/validação cronológicos.

In [7]:
temporal_train_frame = temporal_frames["train"]
temporal_validation_frame = temporal_frames["validation"]
temporal_true_validation_mask = temporal_validation_frame["label"].eq(0).to_numpy()
X_temporal_true_train_raw = temporal_true_train_frame[STYLE_FEATURES]
X_temporal_validation_raw = temporal_validation_frame[STYLE_FEATURES]
y_temporal_validation = temporal_validation_frame["label"].to_numpy(dtype=int)

temporal_imputer = SimpleImputer(strategy="median")
temporal_imputer.fit(X_temporal_true_train_raw)
temporal_true_train_imputed = temporal_imputer.transform(X_temporal_true_train_raw)
temporal_scaler = StandardScaler().fit(temporal_true_train_imputed[:, 1:])


def transform_temporal_style(frame):
    imputed = temporal_imputer.transform(frame[STYLE_FEATURES])
    transformed = np.column_stack((imputed[:, 0], temporal_scaler.transform(imputed[:, 1:])))
    if not np.isfinite(transformed).all() or not np.isin(transformed[:, 0], [0.0, 1.0]).all():
        raise ValueError("Transformação temporal de estilo inválida.")
    return transformed


X_temporal_true_train = transform_temporal_style(temporal_true_train_frame)
X_temporal_validation = transform_temporal_style(temporal_validation_frame)
temporal_novelty_records = []

for candidate_k in NOVELTY_K_CANDIDATES:
    estimator = KMeans(
        n_clusters=candidate_k,
        init="k-means++",
        n_init=KMEANS_N_INIT,
        random_state=RANDOM_STATE,
        algorithm="lloyd",
    )
    estimator.fit(X_temporal_true_train)
    scores = estimator.transform(X_temporal_validation).min(axis=1)
    threshold = float(np.quantile(scores[temporal_true_validation_mask], 0.95))
    temporal_novelty_records.append({
        "method": "KMeansNovelty",
        "k": candidate_k,
        "estimator": estimator,
        "scores_validation": scores,
        "threshold": threshold,
        "validation_metrics": anomaly_metrics(y_temporal_validation, scores, threshold),
        "parameters": {
            "n_clusters": candidate_k, "init": "k-means++", "n_init": KMEANS_N_INIT,
            "random_state": RANDOM_STATE, "algorithm": "lloyd",
        },
    })

temporal_if = IsolationForest(
    n_estimators=300, contamination="auto", random_state=RANDOM_STATE, n_jobs=-1,
).fit(X_temporal_true_train)
temporal_if_scores = -temporal_if.decision_function(X_temporal_validation)
temporal_if_threshold = float(np.quantile(
    temporal_if_scores[temporal_true_validation_mask], 0.95
))
temporal_if_record = {
    "method": "IsolationForest", "parameter_value": 300, "estimator": temporal_if,
    "scores_validation": temporal_if_scores, "threshold": temporal_if_threshold,
    "validation_metrics": anomaly_metrics(y_temporal_validation, temporal_if_scores, temporal_if_threshold),
    "parameters": {"n_estimators": 300, "contamination": "auto", "random_state": RANDOM_STATE, "n_jobs": -1},
}

temporal_lof_records = []
for n_neighbors in LOF_NEIGHBORS:
    estimator = LocalOutlierFactor(
        n_neighbors=n_neighbors, algorithm="auto", metric="minkowski", p=2,
        contamination="auto", novelty=True, n_jobs=1,
    ).fit(X_temporal_true_train)
    scores = -estimator.decision_function(X_temporal_validation)
    threshold = float(np.quantile(scores[temporal_true_validation_mask], 0.95))
    temporal_lof_records.append({
        "method": "LOF", "n_neighbors": n_neighbors, "estimator": estimator,
        "scores_validation": scores, "threshold": threshold,
        "validation_metrics": anomaly_metrics(y_temporal_validation, scores, threshold),
        "parameters": {
            "n_neighbors": n_neighbors, "algorithm": "auto", "metric": "minkowski", "p": 2,
            "contamination": "auto", "novelty": True, "n_jobs": 1,
        },
    })

temporal_ocsvm_records = []
for nu in OCSVM_NU:
    estimator = OneClassSVM(kernel="rbf", gamma="scale", nu=nu).fit(X_temporal_true_train)
    scores = -estimator.decision_function(X_temporal_validation)
    threshold = float(np.quantile(scores[temporal_true_validation_mask], 0.95))
    temporal_ocsvm_records.append({
        "method": "OneClassSVM", "nu": nu, "estimator": estimator,
        "scores_validation": scores, "threshold": threshold,
        "validation_metrics": anomaly_metrics(y_temporal_validation, scores, threshold),
        "parameters": {"kernel": "rbf", "gamma": "scale", "nu": nu},
    })

selected_temporal_novelty = {
    "KMeansNovelty": select_by_validation(temporal_novelty_records, "k"),
    "IsolationForest": temporal_if_record,
    "LOF": select_by_validation(temporal_lof_records, "n_neighbors"),
    "OneClassSVM": select_by_validation(temporal_ocsvm_records, "nu"),
}
for selected in selected_temporal_novelty.values():
    selected["selected"] = True
temporal_frozen_selection = {
    "selection_rule": frozen_selection["selection_rule"],
    "kmeans_novelty": {
        "k": selected_temporal_novelty["KMeansNovelty"]["k"],
        "threshold_q95": selected_temporal_novelty["KMeansNovelty"]["threshold"],
        "validation_roc_auc": selected_temporal_novelty["KMeansNovelty"]["validation_metrics"]["roc_auc"],
        "validation_average_precision": selected_temporal_novelty["KMeansNovelty"]["validation_metrics"]["average_precision"],
        "parameters": selected_temporal_novelty["KMeansNovelty"]["parameters"],
    },
    "isolation_forest": {"threshold_q95": temporal_if_record["threshold"], "parameters": temporal_if_record["parameters"]},
    "lof": {
        "n_neighbors": selected_temporal_novelty["LOF"]["n_neighbors"],
        "threshold_q95": selected_temporal_novelty["LOF"]["threshold"],
        "validation_roc_auc": selected_temporal_novelty["LOF"]["validation_metrics"]["roc_auc"],
        "validation_average_precision": selected_temporal_novelty["LOF"]["validation_metrics"]["average_precision"],
        "parameters": selected_temporal_novelty["LOF"]["parameters"],
    },
    "one_class_svm": {
        "nu": selected_temporal_novelty["OneClassSVM"]["nu"],
        "threshold_q95": selected_temporal_novelty["OneClassSVM"]["threshold"],
        "validation_roc_auc": selected_temporal_novelty["OneClassSVM"]["validation_metrics"]["roc_auc"],
        "validation_average_precision": selected_temporal_novelty["OneClassSVM"]["validation_metrics"]["average_precision"],
        "parameters": selected_temporal_novelty["OneClassSVM"]["parameters"],
    },
    "test_opened": False,
}

temporal_word_vectorizer = TfidfVectorizer(
    ngram_range=(1, 2), min_df=2, sublinear_tf=True,
    token_pattern=r"(?u)\b[^\d\W]{2,}\b", dtype=np.float32,
)
temporal_char_vectorizer = TfidfVectorizer(
    analyzer="char_wb", ngram_range=(3, 5), min_df=3,
    sublinear_tf=True, dtype=np.float32,
)
X_temporal_word_train = temporal_word_vectorizer.fit_transform(temporal_train_frame["texto_trunc"].tolist())
X_temporal_char_train = temporal_char_vectorizer.fit_transform(temporal_train_frame["texto_trunc"].tolist())
X_temporal_topic_train = hstack(
    (X_temporal_word_train, X_temporal_char_train), format="csr", dtype=np.float32
)
X_temporal_topic_validation = hstack((
    temporal_word_vectorizer.transform(temporal_validation_frame["texto_trunc"].tolist()),
    temporal_char_vectorizer.transform(temporal_validation_frame["texto_trunc"].tolist()),
), format="csr", dtype=np.float32)
if not scipy.sparse.issparse(X_temporal_topic_train) or not scipy.sparse.issparse(X_temporal_topic_validation):
    raise AssertionError("A TF-IDF temporal deve permanecer esparsa.")

temporal_topic_records = []
for candidate_k in TOPIC_K_CANDIDATES:
    estimator = KMeans(
        n_clusters=candidate_k, init="k-means++", n_init=TOPIC_N_INIT,
        random_state=RANDOM_STATE, algorithm="lloyd",
    )
    training_clusters = estimator.fit_predict(X_temporal_topic_train)
    validation_clusters = estimator.predict(X_temporal_topic_validation)
    cluster_count = np.unique(validation_clusters).size
    validation_silhouette = float(silhouette_score(
        X_temporal_topic_validation,
        validation_clusters,
        metric="cosine",
        sample_size=min(SILHOUETTE_SAMPLE_SIZE, len(validation_clusters)),
        random_state=RANDOM_STATE,
    )) if 1 < cluster_count < len(validation_clusters) else float("nan")
    temporal_topic_records.append({
        "k": candidate_k, "estimator": estimator,
        "train_cluster_ids": training_clusters,
        "validation_cluster_ids": validation_clusters,
        "validation_silhouette_cosine": validation_silhouette,
        "inertia_train": float(estimator.inertia_),
        "parameters": {
            "n_clusters": candidate_k, "init": "k-means++", "n_init": TOPIC_N_INIT,
            "random_state": RANDOM_STATE, "algorithm": "lloyd",
            "features": "word+char TF-IDF, sparse CSR, temporal train only",
        },
    })
valid_temporal_topic_records = [
    candidate for candidate in temporal_topic_records
    if np.isfinite(candidate["validation_silhouette_cosine"])
]
if not valid_temporal_topic_records:
    raise ValueError("Nenhum k temático temporal formou clusters avaliáveis na validação.")
selected_temporal_topic = max(
    valid_temporal_topic_records,
    key=lambda candidate: (candidate["validation_silhouette_cosine"], -candidate["k"]),
)
selected_temporal_topic["selected"] = True
temporal_frozen_selection["topic_discovery"] = {
    "k": selected_temporal_topic["k"],
    "validation_silhouette_cosine": selected_temporal_topic["validation_silhouette_cosine"],
    "parameters": selected_temporal_topic["parameters"],
    "selection_rule": "maximum validation cosine silhouette; tie -> smaller k",
}
print("Seleção temporal congelada antes da avaliação de teste:")
print(json.dumps(temporal_frozen_selection, ensure_ascii=False, indent=2))
print("TF-IDF temporal esparso:", X_temporal_topic_train.shape, X_temporal_topic_validation.shape)

Seleção temporal congelada antes da avaliação de teste:
{
  "selection_rule": "validation ROC-AUC, then validation AP, then lower k/parameter; ties within 1e-12",
  "kmeans_novelty": {
    "k": 8,
    "threshold_q95": 2.001645334558786,
    "validation_roc_auc": 0.8232408609005153,
    "validation_average_precision": 0.7419871020346718,
    "parameters": {
      "n_clusters": 8,
      "init": "k-means++",
      "n_init": 20,
      "random_state": 42,
      "algorithm": "lloyd"
    }
  },
  "isolation_forest": {
    "threshold_q95": 0.10387356043321005,
    "parameters": {
      "n_estimators": 300,
      "contamination": "auto",
      "random_state": 42,
      "n_jobs": -1
    }
  },
  "lof": {
    "n_neighbors": 10,
    "threshold_q95": 0.2865530690622269,
    "validation_roc_auc": 0.9534004583779189,
    "validation_average_precision": 0.9340787561223917,
    "parameters": {
      "n_neighbors": 10,
      "algorithm": "auto",
      "metric": "minkowski",
      "p": 2,
      "contamin

## Modelos e limiares congelados; agora avaliar os dois testes

Até este ponto os IDs de teste não foram pontuados nem usados para ajustar
transformações. O bloco abaixo abre os dois conjuntos para avaliação final.

In [8]:
test_frame = news.iloc[test_indices].reset_index(drop=True)
X_test_style = transform_style(test_frame)
y_test = test_frame["label"].to_numpy(dtype=int)
temporal_test_frame = news.iloc[temporal_partition_indices["test"]].reset_index(drop=True)
X_temporal_test_style = transform_temporal_style(temporal_test_frame)
y_temporal_test = temporal_test_frame["label"].to_numpy(dtype=int)
frozen_selection["test_opened"] = True
temporal_frozen_selection["test_opened"] = True


def score_detector(model_record, X):
    method = model_record["method"]
    estimator = model_record["estimator"]
    if method == "KMeansNovelty":
        return estimator.transform(X).min(axis=1)
    if method == "IsolationForest":
        return -estimator.decision_function(X)
    if method == "LOF":
        return -estimator.decision_function(X)
    if method == "OneClassSVM":
        return -estimator.decision_function(X)
    raise ValueError(f"Detector desconhecido: {method}")


test_scores_by_method = {}
test_metrics_by_method = {}
for method, model_record in selected_novelty.items():
    scores = score_detector(model_record, X_test_style)
    test_scores_by_method[method] = scores
    test_metrics_by_method[method] = anomaly_metrics(y_test, scores, model_record["threshold"])

temporal_test_scores_by_method = {}
temporal_test_metrics_by_method = {}
for method, model_record in selected_temporal_novelty.items():
    scores = score_detector(model_record, X_temporal_test_style)
    temporal_test_scores_by_method[method] = scores
    temporal_test_metrics_by_method[method] = anomaly_metrics(y_temporal_test, scores, model_record["threshold"])

selected_novelty["KMeansNovelty"]["validation_cluster_ids"] = selected_novelty[
    "KMeansNovelty"
]["estimator"].predict(X_validation)
selected_novelty["KMeansNovelty"]["test_cluster_ids"] = selected_novelty[
    "KMeansNovelty"
]["estimator"].predict(X_test_style)
selected_temporal_novelty["KMeansNovelty"]["validation_cluster_ids"] = selected_temporal_novelty[
    "KMeansNovelty"
]["estimator"].predict(X_temporal_validation)
selected_temporal_novelty["KMeansNovelty"]["test_cluster_ids"] = selected_temporal_novelty[
    "KMeansNovelty"
]["estimator"].predict(X_temporal_test_style)

# Ajusta somente os transformadores de texto no treino; aplica o modelo congelado.
test_texts = test_frame["texto_trunc"].tolist()
X_word_test = word_vectorizer.transform(test_texts)
X_char_test = char_vectorizer.transform(test_texts)
X_topic_test = hstack((X_word_test, X_char_test), format="csr", dtype=np.float32)
selected_topic_model = selected_topic["estimator"]
topic_test_cluster_ids = selected_topic_model.predict(X_topic_test)
topic_test_distances = selected_topic_model.transform(X_topic_test).min(axis=1)
topic_train_cluster_ids = selected_topic["train_cluster_ids"]
topic_validation_cluster_ids = selected_topic["validation_cluster_ids"]
topic_train_distances = selected_topic_model.transform(X_topic_train).min(axis=1)
topic_validation_distances = selected_topic_model.transform(X_topic_validation).min(axis=1)

temporal_test_texts = temporal_test_frame["texto_trunc"].tolist()
X_temporal_word_test = temporal_word_vectorizer.transform(temporal_test_texts)
X_temporal_char_test = temporal_char_vectorizer.transform(temporal_test_texts)
X_temporal_topic_test = hstack(
    (X_temporal_word_test, X_temporal_char_test), format="csr", dtype=np.float32
)
selected_temporal_topic_model = selected_temporal_topic["estimator"]
temporal_topic_train_cluster_ids = selected_temporal_topic["train_cluster_ids"]
temporal_topic_validation_cluster_ids = selected_temporal_topic["validation_cluster_ids"]
temporal_topic_test_cluster_ids = selected_temporal_topic_model.predict(X_temporal_topic_test)
temporal_topic_train_distances = selected_temporal_topic_model.transform(X_temporal_topic_train).min(axis=1)
temporal_topic_validation_distances = selected_temporal_topic_model.transform(
    X_temporal_topic_validation
).min(axis=1)
temporal_topic_test_distances = selected_temporal_topic_model.transform(X_temporal_topic_test).min(axis=1)

## Relatórios externos da descoberta temática

Após congelar k e ajustar o modelo sem labels, os labels externos são usados
somente para ARI/NMI e composição Fake/True. Os exemplos abaixo são IDs, sem
trechos de texto.

In [9]:
word_names = np.char.add("word:", word_vectorizer.get_feature_names_out().astype(str))
char_names = np.char.add("char:", char_vectorizer.get_feature_names_out().astype(str))
feature_names = np.concatenate((word_names, char_names))
center_matrix = selected_topic_model.cluster_centers_


def top_feature_names(center, names, prefix, count=10):
    candidate_indices = np.flatnonzero(np.char.startswith(names, prefix))
    if len(candidate_indices) == 0:
        return []
    selected_indices = candidate_indices[np.argsort(center[candidate_indices])[-count:][::-1]]
    return [str(names[index]).removeprefix(prefix) for index in selected_indices]


topic_cluster_profiles = []
for cluster_id in range(selected_topic_k):
    train_cluster_rows = np.flatnonzero(topic_train_cluster_ids == cluster_id)
    if train_cluster_rows.size:
        nearest_order = train_cluster_rows[np.argsort(topic_train_distances[train_cluster_rows])[:5]]
        nearest_record_ids = train_frame.iloc[nearest_order]["record_id"].astype(str).tolist()
    else:
        nearest_record_ids = []
    topic_cluster_profiles.append({
        "cluster_id": int(cluster_id),
        "train_size": int(np.sum(topic_train_cluster_ids == cluster_id)),
        "validation_size": int(np.sum(topic_validation_cluster_ids == cluster_id)),
        "test_size": int(np.sum(topic_test_cluster_ids == cluster_id)),
        "representative_word_terms": top_feature_names(center_matrix[cluster_id], feature_names, "word:"),
        "representative_char_ngrams": top_feature_names(center_matrix[cluster_id], feature_names, "char:"),
        "nearest_train_record_ids": nearest_record_ids,
    })


def topic_external_metrics(
    labels,
    cluster_ids,
    distances,
    sparse_matrix,
    partition_name,
    protocol="canonical_random_group",
    selected_k=None,
    inertia_train=None,
    stability_value=None,
    selection_metric="validation cosine silhouette; labels not used",
):
    labels = np.asarray(labels, dtype=int)
    cluster_ids = np.asarray(cluster_ids, dtype=int)
    cluster_sizes = Counter(cluster_ids.tolist())
    composition = {}
    for cluster_id in sorted(cluster_sizes):
        mask = cluster_ids == cluster_id
        composition[str(cluster_id)] = {
            "total": int(mask.sum()),
            "true_0": int(np.sum(labels[mask] == 0)),
            "fake_1": int(np.sum(labels[mask] == 1)),
        }
    if 1 < np.unique(cluster_ids).size < len(cluster_ids):
        internal_silhouette = float(silhouette_score(
            sparse_matrix,
            cluster_ids,
            metric="cosine",
            sample_size=min(SILHOUETTE_SAMPLE_SIZE, len(cluster_ids)),
            random_state=RANDOM_STATE,
        ))
    else:
        internal_silhouette = float("nan")
    return {
        "protocol": protocol,
        "track": "topic_discovery",
        "method": "KMeansTopic",
        "partition": partition_name,
        "n_samples": int(len(labels)),
        "n_true": int(np.sum(labels == 0)),
        "n_fake": int(np.sum(labels == 1)),
        "k": int(selected_topic_k if selected_k is None else selected_k),
        "cluster_count": int(len(cluster_sizes)),
        "cluster_sizes_json": json.dumps(dict(sorted(cluster_sizes.items())), sort_keys=True),
        "cluster_composition_json": json.dumps(composition, sort_keys=True),
        "inertia_train": float(selected_topic["inertia_train"] if inertia_train is None else inertia_train),
        "mean_squared_distance_to_centroid": float(np.mean(np.square(distances))),
        "silhouette_cosine": internal_silhouette,
        "ari": float(adjusted_rand_score(labels, cluster_ids)),
        "nmi": float(normalized_mutual_info_score(labels, cluster_ids)),
        "stability_mean_pairwise_ari_train": stability_mean_ari if stability_value is None else stability_value,
        "selection_metric": selection_metric,
        "selected": True,
    }


topic_external_records = [
    topic_external_metrics(
        train_frame["label"].to_numpy(dtype=int), topic_train_cluster_ids, topic_train_distances,
        X_topic_train, "train",
    ),
    topic_external_metrics(
        validation_frame["label"].to_numpy(dtype=int), topic_validation_cluster_ids,
        topic_validation_distances, X_topic_validation, "validation",
    ),
    topic_external_metrics(
        test_frame["label"].to_numpy(dtype=int), topic_test_cluster_ids, topic_test_distances,
        X_topic_test, "test",
    ),
]

temporal_topic_external_records = [
    topic_external_metrics(
        temporal_train_frame["label"].to_numpy(dtype=int), temporal_topic_train_cluster_ids,
        temporal_topic_train_distances, X_temporal_topic_train, "train",
        protocol="secondary_temporal_group", selected_k=selected_temporal_topic["k"],
        inertia_train=selected_temporal_topic["inertia_train"], stability_value=np.nan,
        selection_metric="maximum temporal-validation cosine silhouette; labels not used",
    ),
    topic_external_metrics(
        temporal_validation_frame["label"].to_numpy(dtype=int), temporal_topic_validation_cluster_ids,
        temporal_topic_validation_distances, X_temporal_topic_validation, "validation",
        protocol="secondary_temporal_group", selected_k=selected_temporal_topic["k"],
        inertia_train=selected_temporal_topic["inertia_train"], stability_value=np.nan,
        selection_metric="maximum temporal-validation cosine silhouette; labels not used",
    ),
    topic_external_metrics(
        temporal_test_frame["label"].to_numpy(dtype=int), temporal_topic_test_cluster_ids,
        temporal_topic_test_distances, X_temporal_topic_test, "test",
        protocol="secondary_temporal_group", selected_k=selected_temporal_topic["k"],
        inertia_train=selected_temporal_topic["inertia_train"], stability_value=np.nan,
        selection_metric="maximum temporal-validation cosine silhouette; labels not used",
    ),
]

## Artefatos sem texto bruto

`metrics.csv` registra candidatos e resultados por método/partição;
`predictions.csv` usa IDs, labels apenas para avaliação e scores, nunca texto;
`run_manifest.json` guarda hash, parâmetros e os IDs de cada partição.

In [10]:
run_base = project_root / "machine-learning" / "outputs" / "model-comparison"
run_base.mkdir(parents=True, exist_ok=True)
run_stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
run_id = f"kmeans-canonical-{run_stamp}"
run_path = run_base / run_id
suffix = 2
while run_path.exists():
    run_id = f"kmeans-canonical-{run_stamp}-{suffix}"
    run_path = run_base / run_id
    suffix += 1
run_path.mkdir(parents=True, exist_ok=False)

metric_columns = [
    "run_id", "protocol", "track", "method", "partition", "seed", "status", "selected",
    "parameters_json", "n_samples", "n_true", "n_fake", "roc_auc", "average_precision",
    "macro_f1", "balanced_accuracy", "precision_fake", "recall_fake", "f1_fake", "fpr_true",
    "accuracy", "threshold_q95", "q95_source", "validation_true_alert_rate", "k",
    "inertia_train", "validation_silhouette_cosine", "mean_squared_distance_to_centroid",
    "silhouette_cosine", "ari", "nmi", "cluster_count", "cluster_sizes_json",
    "cluster_composition_json", "stability_mean_pairwise_ari_train", "selection_metric",
]
metric_records = []


def add_metric_record(record):
    output = {column: np.nan for column in metric_columns}
    output.update({
        "run_id": run_id,
        "protocol": "canonical_random_group",
        "seed": RANDOM_STATE,
        "status": "executed",
    })
    output.update(record)
    metric_records.append(output)


for candidate in novelty_validation_records:
    add_metric_record({
        "track": "novelty_detection",
        "method": "KMeansNovelty",
        "partition": "validation",
        "selected": candidate is selected_kmeans_novelty,
        "parameters_json": json.dumps(candidate["parameters"], sort_keys=True),
        "k": candidate["k"],
        **candidate["validation_metrics"],
        "selection_metric": frozen_selection["selection_rule"],
    })
for candidate in [if_record] + lof_records + ocsvm_records:
    add_metric_record({
        "track": "novelty_detection",
        "method": candidate["method"],
        "partition": "validation",
        "selected": candidate is if_record or candidate is selected_lof or candidate is selected_ocsvm,
        "parameters_json": json.dumps(candidate["parameters"], sort_keys=True),
        "k": np.nan,
        **candidate["validation_metrics"],
        "selection_metric": frozen_selection["selection_rule"],
    })

for method, model_record in selected_novelty.items():
    add_metric_record({
        "track": "novelty_detection",
        "method": method,
        "partition": "test",
        "selected": True,
        "parameters_json": json.dumps(model_record["parameters"], sort_keys=True),
        "k": model_record.get("k", np.nan),
        **test_metrics_by_method[method],
        "selection_metric": frozen_selection["selection_rule"],
    })

for candidate in topic_candidate_records:
    add_metric_record({
        "track": "topic_k_selection",
        "method": "KMeansTopic",
        "partition": "validation",
        "selected": candidate is selected_topic,
        "parameters_json": json.dumps(candidate["parameters"], sort_keys=True),
        "k": candidate["k"],
        "inertia_train": candidate["inertia_train"],
        "validation_silhouette_cosine": candidate["validation_silhouette_cosine"],
        "selection_metric": "maximum validation cosine silhouette; tie -> smaller k",
    })
for record in topic_external_records:
    add_metric_record(record)
for pair in stability_pairs:
    add_metric_record({
        "track": "topic_stability",
        "method": "KMeansTopic",
        "partition": "train",
        "seed": pair["seed_right"],
        "selected": True,
        "parameters_json": json.dumps({
            "k": selected_topic_k,
            "seed_left": pair["seed_left"],
            "seed_right": pair["seed_right"],
            "n_init": TOPIC_N_INIT,
        }, sort_keys=True),
        "k": selected_topic_k,
        "ari": pair["ari"],
        "stability_mean_pairwise_ari_train": stability_mean_ari,
        "selection_metric": "external adjustment of unlabeled train assignments across seeds",
    })

for candidate in temporal_novelty_records:
    add_metric_record({
        "protocol": "secondary_temporal_group",
        "track": "novelty_detection",
        "method": "KMeansNovelty",
        "partition": "validation",
        "selected": candidate is selected_temporal_novelty["KMeansNovelty"],
        "parameters_json": json.dumps(candidate["parameters"], sort_keys=True),
        "k": candidate["k"],
        **candidate["validation_metrics"],
        "selection_metric": temporal_frozen_selection["selection_rule"],
    })
for candidate in [temporal_if_record] + temporal_lof_records + temporal_ocsvm_records:
    add_metric_record({
        "protocol": "secondary_temporal_group",
        "track": "novelty_detection",
        "method": candidate["method"],
        "partition": "validation",
        "selected": candidate is temporal_if_record
        or candidate is selected_temporal_novelty["LOF"]
        or candidate is selected_temporal_novelty["OneClassSVM"],
        "parameters_json": json.dumps(candidate["parameters"], sort_keys=True),
        "k": candidate.get("k", np.nan),
        **candidate["validation_metrics"],
        "selection_metric": temporal_frozen_selection["selection_rule"],
    })
for method, model_record in selected_temporal_novelty.items():
    add_metric_record({
        "protocol": "secondary_temporal_group",
        "track": "novelty_detection",
        "method": method,
        "partition": "test",
        "selected": True,
        "parameters_json": json.dumps(model_record["parameters"], sort_keys=True),
        "k": model_record.get("k", np.nan),
        **temporal_test_metrics_by_method[method],
        "selection_metric": temporal_frozen_selection["selection_rule"],
    })
for candidate in temporal_topic_records:
    add_metric_record({
        "protocol": "secondary_temporal_group",
        "track": "topic_k_selection",
        "method": "KMeansTopic",
        "partition": "validation",
        "selected": candidate is selected_temporal_topic,
        "parameters_json": json.dumps(candidate["parameters"], sort_keys=True),
        "k": candidate["k"],
        "inertia_train": candidate["inertia_train"],
        "validation_silhouette_cosine": candidate["validation_silhouette_cosine"],
        "selection_metric": "maximum temporal-validation cosine silhouette; tie -> smaller k",
    })
for record in temporal_topic_external_records:
    add_metric_record(record)

metrics_frame = pd.DataFrame.from_records(metric_records, columns=metric_columns)
metrics_frame.to_csv(run_path / "metrics.csv", index=False, encoding="utf-8")

prediction_records = []
for method, model_record in selected_novelty.items():
    for partition_name, frame, style_matrix in (
        ("validation", validation_frame, X_validation),
        ("test", test_frame, X_test_style),
    ):
        scores = model_record["scores_validation"] if partition_name == "validation" else test_scores_by_method[method]
        if method == "KMeansNovelty":
            clusters = model_record["validation_cluster_ids"] if partition_name == "validation" else model_record["test_cluster_ids"]
        else:
            clusters = [None] * len(frame)
        decisions = scores >= model_record["threshold"]
        for index, row in enumerate(frame.itertuples(index=False)):
            prediction_records.append({
                "record_id": row.record_id,
                "group_id": row.group_id,
                "protocol": "canonical_random_group",
                "partition": partition_name,
                "label_for_evaluation_only": int(row.label),
                "track": "novelty_detection",
                "method": method,
                "cluster_id": None if clusters[index] is None else int(clusters[index]),
                "score": float(scores[index]),
                "score_type": "distance_to_nearest_centroid" if method == "KMeansNovelty" else "anomaly_score",
                "threshold_q95": float(model_record["threshold"]),
                "decision_fake": bool(decisions[index]),
            })

for partition_name, frame, clusters, distances in (
    ("train", train_frame, topic_train_cluster_ids, topic_train_distances),
    ("validation", validation_frame, topic_validation_cluster_ids, topic_validation_distances),
    ("test", test_frame, topic_test_cluster_ids, topic_test_distances),
):
    for index, row in enumerate(frame.itertuples(index=False)):
        prediction_records.append({
            "record_id": row.record_id,
            "group_id": row.group_id,
            "protocol": "canonical_random_group",
            "partition": partition_name,
            "label_for_evaluation_only": int(row.label),
            "track": "topic_discovery",
            "method": "KMeansTopic",
            "cluster_id": int(clusters[index]),
            "score": float(distances[index]),
            "score_type": "distance_to_nearest_centroid",
            "threshold_q95": None,
            "decision_fake": None,
        })

for method, model_record in selected_temporal_novelty.items():
    for partition_name, frame in (
        ("validation", temporal_validation_frame),
        ("test", temporal_test_frame),
    ):
        scores = model_record["scores_validation"] if partition_name == "validation" else temporal_test_scores_by_method[method]
        if method == "KMeansNovelty":
            clusters = model_record["validation_cluster_ids"] if partition_name == "validation" else model_record["test_cluster_ids"]
        else:
            clusters = [None] * len(frame)
        decisions = scores >= model_record["threshold"]
        for index, row in enumerate(frame.itertuples(index=False)):
            prediction_records.append({
                "record_id": row.record_id,
                "group_id": row.group_id,
                "protocol": "secondary_temporal_group",
                "partition": partition_name,
                "label_for_evaluation_only": int(row.label),
                "track": "novelty_detection",
                "method": method,
                "cluster_id": None if clusters[index] is None else int(clusters[index]),
                "score": float(scores[index]),
                "score_type": "distance_to_nearest_centroid" if method == "KMeansNovelty" else "anomaly_score",
                "threshold_q95": float(model_record["threshold"]),
                "decision_fake": bool(decisions[index]),
            })

for partition_name, frame, clusters, distances in (
    ("train", temporal_train_frame, temporal_topic_train_cluster_ids, temporal_topic_train_distances),
    ("validation", temporal_validation_frame, temporal_topic_validation_cluster_ids, temporal_topic_validation_distances),
    ("test", temporal_test_frame, temporal_topic_test_cluster_ids, temporal_topic_test_distances),
):
    for index, row in enumerate(frame.itertuples(index=False)):
        prediction_records.append({
            "record_id": row.record_id,
            "group_id": row.group_id,
            "protocol": "secondary_temporal_group",
            "partition": partition_name,
            "label_for_evaluation_only": int(row.label),
            "track": "topic_discovery",
            "method": "KMeansTopic",
            "cluster_id": int(clusters[index]),
            "score": float(distances[index]),
            "score_type": "distance_to_nearest_centroid",
            "threshold_q95": None,
            "decision_fake": None,
        })

predictions_frame = pd.DataFrame.from_records(prediction_records)
if predictions_frame.duplicated(["protocol", "track", "method", "partition", "record_id"]).any():
    raise AssertionError("Predições duplicadas dentro do mesmo protocolo/método/partição.")
predictions_frame.to_csv(run_path / "predictions.csv", index=False, encoding="utf-8")

profiles_frame = pd.DataFrame.from_records(topic_cluster_profiles)
for column in ("representative_word_terms", "representative_char_ngrams", "nearest_train_record_ids"):
    profiles_frame[column] = profiles_frame[column].map(lambda value: json.dumps(value, ensure_ascii=False))
profiles_frame.to_csv(run_path / "topic_cluster_profiles.csv", index=False, encoding="utf-8")

version_names = ["numpy", "pandas", "scipy", "scikit-learn"]
versions = {name: version(name) for name in version_names}
manifest = {
    "run_id": run_id,
    "status": "executed",
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "corpus": {
        "name": "Fake.br-Corpus",
        "revision": CORPUS_REVISION,
        "url": CORPUS_URL,
        "archive_sha256": archive_sha256,
        "historical_archive_sha256": HISTORICAL_ARCHIVE_SHA256,
        "historical_sha256_matches": archive_sha256 == HISTORICAL_ARCHIVE_SHA256,
        "record_count": int(len(news)),
        "labels": {"0": "True", "1": "Fake"},
    },
    "identity": {
        "record_id": "source folder + original filename stem",
        "group_id": "connected components of aligned Fake/True same-stem pairs and exact normalized full-text duplicates",
        "groups_count": int(news["group_id"].nunique()),
        "aligned_pair_stems": int(len(aligned_pair_stems)),
        "split_group_overlap": False,
    },
    "partitions": {
        name: {
            "record_ids": sorted(partition_ids[name]),
            "group_ids": sorted(partition_groups[name]),
            "counts": {
                "records": int(len(partition_ids[name])),
                "groups": int(len(partition_groups[name])),
                "true_0": int(news.iloc[partition_indices[name]]["label"].eq(0).sum()),
                "fake_1": int(news.iloc[partition_indices[name]]["label"].eq(1).sum()),
            },
        }
        for name in ("train", "validation", "test")
    },
    "split": {
        "method": "StratifiedGroupKFold",
        "n_splits": N_SPLITS,
        "shuffle": True,
        "seed": RANDOM_STATE,
        "fold_0": "test",
        "fold_1": "validation",
        "other_folds": "train",
        "secondary_holdout_audit": secondary_audit,
        "secondary_holdout_status": "temporal secondary executed; strict source holdout not run because the aligned-source graph has one component and domains are class-separated",
        "secondary_temporal": {
            "method": "strict chronological group holdout over groups with all publication dates parseable",
            "group_time": "latest publication date among records in the aligned/deduplicated group",
            "date_parser": "DD/MM/YYYY, YYYY-MM-DD, or Portuguese D de month de YYYY; NFKD month normalization",
            "date_boundaries_utc": {
                "train_end_exclusive": temporal_train_boundary.isoformat(),
                "test_start_inclusive": temporal_test_boundary.isoformat(),
            },
            "excluded_group_ids": sorted(
                set(news["group_id"]) - set(temporal_group_frame["group_id"])
            ),
            "partitions": {
                name: {
                    "record_ids": sorted(temporal_partition_ids[name]),
                    "group_ids": sorted(temporal_group_sets[name]),
                    "counts": {
                        "records": int(len(temporal_partition_ids[name])),
                        "groups": int(len(temporal_group_sets[name])),
                        "true_0": int(temporal_frames[name]["label"].eq(0).sum())
                        if name in temporal_frames else int(temporal_test_frame["label"].eq(0).sum()),
                        "fake_1": int(temporal_frames[name]["label"].eq(1).sum())
                        if name in temporal_frames else int(temporal_test_frame["label"].eq(1).sum()),
                        "first_group_date_utc": temporal_group_frame.loc[
                            temporal_group_frame["group_id"].isin(temporal_group_sets[name]), "group_time"
                        ].min().isoformat(),
                        "last_group_date_utc": temporal_group_frame.loc[
                            temporal_group_frame["group_id"].isin(temporal_group_sets[name]), "group_time"
                        ].max().isoformat(),
                    },
                }
                for name in ("train", "validation", "test")
            },
        },
    },
    "environment": {
        "python": platform.python_version(),
        "packages": versions,
        "platform": platform.platform(),
    },
    "features": {
        "novelty_style": STYLE_FEATURES,
        "style_text_prefix_characters": CHARACTER_LIMIT,
        "style_normalization": ["Unicode NFKC", "remove leading BOM"],
        "authorship": "binary raw 0/1; not standardized; maximum pairwise contribution 1",
        "imputation": "median fit only on True-train",
        "scaling": "StandardScaler fit only on the five non-author style features from True-train",
        "topic_text": "texto_trunc, first 200 whitespace-separated words",
        "topic_tfidf": {
            "word": {"ngram_range": [1, 2], "min_df": 2, "sublinear_tf": True, "dtype": "float32"},
            "character": {"analyzer": "char_wb", "ngram_range": [3, 5], "min_df": 3, "sublinear_tf": True, "dtype": "float32"},
            "fit_partition": "train only",
            "matrix_format": "scipy CSR sparse",
        },
    },
    "novelty_detection": {
        "fit_records": "True rows from canonical train only",
        "score_direction": "higher means greater deviation",
        "threshold": "q95 of True-validation scores; method-specific, numpy quantile default linear",
        "selection": frozen_selection,
        "selection_validation_label_assisted": True,
        "selection_test_independent": True,
        "secondary_temporal_selection": temporal_frozen_selection,
        "candidates": {
            "kmeans_k": list(NOVELTY_K_CANDIDATES),
            "kmeans_init": "k-means++",
            "kmeans_n_init": KMEANS_N_INIT,
            "kmeans_seed": RANDOM_STATE,
            "lof_n_neighbors": list(LOF_NEIGHBORS),
            "ocsvm_nu": list(OCSVM_NU),
            "isolation_forest_n_estimators": 300,
        },
        "metrics": [
            "macro-F1", "balanced accuracy", "precision/recall/F1 Fake", "FPR True", "accuracy", "ROC-AUC", "AP",
        ],
    },
    "topic_discovery": {
        "labels_used_for_fit_or_selection": False,
        "candidate_k": list(TOPIC_K_CANDIDATES),
        "selection": "highest validation cosine silhouette; tie -> smaller k",
        "selected_k": int(selected_topic_k),
        "selected_parameters": selected_topic["parameters"],
        "stability_seeds": list(STABILITY_SEEDS),
        "stability_pairwise_ari_train": stability_pairs,
        "stability_mean_pairwise_ari_train": stability_mean_ari,
        "labels_used_after_freeze_for": ["ARI", "NMI", "cluster composition"],
        "no_classification_decision": True,
        "secondary_temporal": {
            "candidate_k": list(TOPIC_K_CANDIDATES),
            "selected_k": int(selected_temporal_topic["k"]),
            "selection": "highest temporal-validation cosine silhouette; tie -> smaller k",
            "selected_parameters": selected_temporal_topic["parameters"],
            "labels_used_for_fit_or_selection": False,
            "labels_used_after_freeze_for": ["ARI", "NMI", "cluster composition"],
        },
    },
    "comparison": {
        "if_lof_one_class_svm_reexecuted": True,
        "same_record_ids_and_partition_within_each_protocol": True,
        "baselines_reexecuted_in_each_protocol": True,
        "old_split_metrics_used_in_ranking": False,
        "source_or_api_cost": "none",
    },
    "artifacts": ["metrics.csv", "predictions.csv", "topic_cluster_profiles.csv", "run_manifest.json"],
}
(run_path / "run_manifest.json").write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8"
)

print("Run:", run_id)
print("Output:", run_path)
print("Novelty metrics, validation selection and frozen test:")
novelty_summary = metrics_frame.loc[
    (metrics_frame["track"] == "novelty_detection") & metrics_frame["partition"].isin(["validation", "test"]),
    ["protocol", "method", "partition", "k", "roc_auc", "average_precision", "macro_f1", "balanced_accuracy", "precision_fake", "recall_fake", "f1_fake", "fpr_true", "accuracy"],
]
print(novelty_summary.to_string(index=False, float_format=lambda value: f"{value:.4f}"))
print("Topic metrics (labels external after freeze):")
topic_summary = pd.DataFrame(topic_external_records)[
    ["protocol", "partition", "k", "n_samples", "cluster_count", "cluster_sizes_json", "silhouette_cosine", "ari", "nmi", "stability_mean_pairwise_ari_train"]
]
print(topic_summary.to_string(index=False, float_format=lambda value: f"{value:.4f}"))
print("Topic metrics, holdout temporal secundário:")
temporal_topic_summary = pd.DataFrame(temporal_topic_external_records)[
    ["protocol", "partition", "k", "n_samples", "cluster_count", "cluster_sizes_json", "silhouette_cosine", "ari", "nmi"]
]
print(temporal_topic_summary.to_string(index=False, float_format=lambda value: f"{value:.4f}"))
print("Perfis temáticos (termos e IDs, sem textos):")
print(profiles_frame.to_string(index=False))

Run: kmeans-canonical-20260924T003936Z
Output: C:\Users\CUL7CA\Desktop\Eldorado\olimpo-fake-news-ai\machine-learning\outputs\model-comparison\kmeans-canonical-20260924T003936Z
Novelty metrics, validation selection and frozen test:
                protocol          method  partition      k  roc_auc  average_precision  macro_f1  balanced_accuracy  precision_fake  recall_fake  f1_fake  fpr_true  accuracy
  canonical_random_group   KMeansNovelty validation 1.0000   0.6563             0.6245    0.4512             0.5424          0.7293       0.1347   0.2274    0.0500    0.5424
  canonical_random_group   KMeansNovelty validation 2.0000   0.7278             0.6678    0.4402             0.5361          0.7097       0.1222   0.2085    0.0500    0.5361
  canonical_random_group   KMeansNovelty validation 4.0000   0.7599             0.6913    0.4427             0.5375          0.7143       0.1250   0.2128    0.0500    0.5375
  canonical_random_group   KMeansNovelty validation 8.0000   0.8351      

## Integridade final

Confere os arquivos mínimos, identidade de IDs/partições e ausência de coluna
com texto bruto. A saída do notebook mostra somente contagens e resultados.

In [11]:
required_artifacts = ("metrics.csv", "predictions.csv", "run_manifest.json", "topic_cluster_profiles.csv")
for artifact_name in required_artifacts:
    artifact_path = run_path / artifact_name
    if not artifact_path.exists() or artifact_path.stat().st_size == 0:
        raise AssertionError(f"Artefato ausente ou vazio: {artifact_path}")
if "text" in predictions_frame.columns or "texto_trunc" in predictions_frame.columns:
    raise AssertionError("Texto bruto/truncado não pode entrar em predictions.csv.")
if predictions_frame["record_id"].isna().any() or predictions_frame["group_id"].isna().any():
    raise AssertionError("IDs ausentes em predictions.csv.")
print("Artefatos conferidos:", {name: (run_path / name).stat().st_size for name in required_artifacts})

Artefatos conferidos: {'metrics.csv': 28937, 'predictions.csv': 5856583, 'run_manifest.json': 620137, 'topic_cluster_profiles.csv': 2509}
